# Lesson 04 — Face Detection (Pre-trained Model)

> **Goal:** Detect faces on real images using a pre-trained model. Fast win before we build our own.  
> **What you already know:** CNNs, filters, feature maps, why deep networks work.  
> **What's new:** Transfer learning, OpenCV, bounding boxes, confidence scores.

---

## Why Pre-trained First?

Training a face detector from scratch needs:
- Thousands of labeled face images
- Hours of compute
- Careful tuning

A pre-trained model has already done all of that. We just load it and run it.  
This gives us a working detector **today** — and a clear target to match when we build our own.

We'll use **OpenCV's built-in face detector** — it uses a method called Haar Cascades,  
which predates deep learning but is fast, accurate, and ships with OpenCV.

Then we'll use a **deep learning based detector** using a pre-trained TensorFlow model.

---

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import urllib.request

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

---
## Step 1 — Get a Test Image

We'll download a public domain image with faces to test on.

In [ ]:
# Download a test image (public domain photo with faces)
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ec/Mona_Lisa%2C_by_Leonardo_da_Vinci%2C_from_C2RMF_retouched.jpg/402px-Mona_Lisa%2C_by_Leonardo_da_Vinci%2C_from_C2RMF_retouched.jpg"
urllib.request.urlretrieve(url, "test_image.jpg")

# Load with OpenCV
img_bgr = cv2.imread("test_image.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)  # OpenCV loads BGR, matplotlib needs RGB

plt.imshow(img_rgb)
plt.title(f"Test image — shape: {img_rgb.shape}")
plt.axis('off')
plt.show()

**Note:** OpenCV loads images as BGR (Blue, Green, Red) not RGB.  
Always convert with `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` before displaying with matplotlib.  
This is a very common bug — keep it in mind.

---
## Step 2 — Haar Cascade Detector (Classical, Pre-deep learning)

Haar Cascades were invented in 2001 by Viola and Jones.  
They use hand-crafted features (not learned by a CNN) and a cascade of classifiers.  
Still widely used because they're fast and require no GPU.

Think of it as: many simple yes/no questions applied in sequence —  
*"Is there a dark region above a light region here? No → definitely not a face, skip."*  
Most image regions get rejected early. Only face-like regions pass all stages.

In [ ]:
# Load the pre-trained Haar Cascade for frontal faces
# This XML file ships with OpenCV — no download needed
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Convert to grayscale — Haar works on grayscale
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Detect faces
# scaleFactor: how much to shrink image each pass (1.1 = 10% smaller each time)
# minNeighbors: how many overlapping detections needed to confirm a face
# minSize: minimum face size in pixels
faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30, 30)
)

print(f"Faces detected: {len(faces)}")
if len(faces) > 0:
    print("Bounding boxes (x, y, width, height):")
    for face in faces:
        print(" ", face)

In [ ]:
# Draw bounding boxes on the image
img_result = img_rgb.copy()

for (x, y, w, h) in faces:
    cv2.rectangle(
        img_result,
        (x, y),           # top-left corner
        (x + w, y + h),   # bottom-right corner
        (0, 255, 0),       # green color (RGB)
        thickness=3
    )

plt.figure(figsize=(8, 10))
plt.imshow(img_result)
plt.title(f"Haar Cascade — {len(faces)} face(s) detected")
plt.axis('off')
plt.show()

---
## Step 3 — Understanding Bounding Boxes

A bounding box is just 4 numbers: `(x, y, width, height)`

```
(x, y) ──────────────┐
  │                  │
  │    FACE          │  height
  │                  │
  └──────────────────┘
         width
```

- `x, y` = top-left corner pixel coordinates
- `width, height` = size of the box

When we build our own detector in Lesson 05, the network will need to **predict these 4 numbers** — that's called regression. That's why the last layer won't just be `Dense(1)` for face/no-face — it'll also have a branch that outputs `Dense(4)` for the box coordinates.

In [ ]:
# Crop and display each detected face
if len(faces) > 0:
    fig, axes = plt.subplots(1, len(faces), figsize=(4 * len(faces), 4))
    if len(faces) == 1:
        axes = [axes]
    for i, (x, y, w, h) in enumerate(faces):
        face_crop = img_rgb[y:y+h, x:x+w]
        axes[i].imshow(face_crop)
        axes[i].set_title(f"Face {i+1}: {w}×{h}px")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

---
## Step 4 — Test on Your Own Image

Try it on any image. Put a photo in the same folder as this notebook and run:

In [ ]:
def detect_faces(image_path, scale=1.1, neighbors=5):
    """
    Detect faces in an image and return the result with bounding boxes drawn.
    
    Args:
        image_path: path to image file
        scale: scaleFactor for detection (try 1.05 to 1.3)
        neighbors: minNeighbors (higher = fewer false positives)
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f"Could not load image: {image_path}")
        return

    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, scaleFactor=scale, minNeighbors=neighbors, minSize=(30, 30))

    result = img_rgb.copy()
    for (x, y, w, h) in faces:
        cv2.rectangle(result, (x, y), (x+w, y+h), (0, 255, 0), 3)
        cv2.putText(result, 'Face', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    plt.figure(figsize=(10, 8))
    plt.imshow(result)
    plt.title(f"{len(faces)} face(s) detected")
    plt.axis('off')
    plt.show()

    return faces

# --- Try it on your own image ---
# detect_faces("your_photo.jpg")

# For now, run it on our test image
detect_faces("test_image.jpg")

---
## Step 5 — Limitations of Haar Cascades

Try these and observe what breaks:

In [ ]:
# Test 1: More aggressive detection (lower neighbors = more detections, more false positives)
detect_faces("test_image.jpg", neighbors=2)

# Test 2: Very conservative (high neighbors = fewer detections, fewer false positives)
detect_faces("test_image.jpg", neighbors=10)

**Observations:**
- Haar Cascades struggle with: tilted faces, side profiles, dark images, small faces
- They work best on: frontal, well-lit, reasonably sized faces
- False positives (boxes on non-faces) are common at low `minNeighbors`

This is **exactly why deep learning detectors exist** — they learn from data rather than hand-crafted rules, so they generalize much better.

In Lesson 05 we build one ourselves.

---
## Step 6 — Deep Learning Detector (TF + OpenCV DNN)

OpenCV also ships with a deep learning based face detector — a small CNN trained on faces.  
It's far more accurate than Haar, especially on angled or partially occluded faces.

In [ ]:
# Download the pre-trained model files
prototxt_url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
model_url    = "https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"

urllib.request.urlretrieve(prototxt_url, "deploy.prototxt")
urllib.request.urlretrieve(model_url,    "face_detector.caffemodel")

print("Model files downloaded.")

In [ ]:
# Load the deep learning face detector
net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "face_detector.caffemodel")

def detect_faces_deep(image_path, confidence_threshold=0.5):
    """
    Deep learning based face detection.
    Returns confidence scores — not just yes/no.
    """
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_bgr.shape[:2]

    # Preprocess: resize to 300x300, normalize
    # This is what the model was trained on
    blob = cv2.dnn.blobFromImage(
        cv2.resize(img_bgr, (300, 300)),
        scalefactor=1.0,
        size=(300, 300),
        mean=(104.0, 177.0, 123.0)  # mean BGR values subtracted during training
    )

    # Forward pass
    net.setInput(blob)
    detections = net.forward()   # shape: (1, 1, 200, 7)

    result = img_rgb.copy()
    face_count = 0

    # Each detection: [_, _, confidence, x1, y1, x2, y2]
    # Coordinates are normalized 0-1, multiply by image size to get pixels
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]

        if confidence > confidence_threshold:
            x1 = int(detections[0, 0, i, 3] * w)
            y1 = int(detections[0, 0, i, 4] * h)
            x2 = int(detections[0, 0, i, 5] * w)
            y2 = int(detections[0, 0, i, 6] * h)

            cv2.rectangle(result, (x1, y1), (x2, y2), (0, 255, 0), 3)
            label = f"{confidence*100:.1f}%"
            cv2.putText(result, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            face_count += 1

    plt.figure(figsize=(8, 10))
    plt.imshow(result)
    plt.title(f"Deep Learning Detector — {face_count} face(s) detected")
    plt.axis('off')
    plt.show()

detect_faces_deep("test_image.jpg")

**Notice the confidence score on each box.**  
This is the network's sigmoid output — the probability that this region contains a face.  
That's exactly what our own model will output in Lesson 05.

Also notice the bounding boxes are stored as `(x1, y1, x2, y2)` here — two corners — rather than `(x, y, w, h)`. Both are valid formats. We'll use `(x1, y1, x2, y2)` in our own detector.

---
## ✅ Lesson 04 — Summary

| Concept | What you learned |
|---------|------------------|
| Pre-trained model | Load and run without any training |
| Haar Cascade | Classical detector — fast, hand-crafted features, struggles with angles |
| Deep learning detector | CNN-based, outputs confidence scores, much more robust |
| Bounding box | 4 numbers: `(x, y, w, h)` or `(x1, y1, x2, y2)` |
| Confidence score | Sigmoid output — probability this region is a face |
| BGR vs RGB | OpenCV loads BGR — always convert before displaying |
| `cv2.rectangle` | Draw bounding boxes on images |

---

## Think about these before Lesson 05:

1. The deep detector resizes every image to **300×300** before passing to the network. Why?
2. The output has both a **confidence score** and **box coordinates**. That means the network has two jobs at once — what does that tell you about the output layer design?
3. We subtracted mean BGR values `(104, 177, 123)` during preprocessing. Why subtract the mean?

---

## 🚀 Next: Lesson 05 — Build Your Own Face Detector
We build a CNN from scratch that outputs a confidence score + bounding box.  
Everything from Lessons 01–04 comes together here.